# 投资组合回测框架使用示例

本 notebook 演示如何使用可扩展的投资组合回测框架

In [1]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np

from portfolio_backtest import (
    BacktestEngine,
    RiskParityStrategy,
    MeanVarianceStrategy
)
from portfolio_backtest.visualization import BacktestVisualizer
from portfolio_backtest.utils import load_price_data

## 1. 加载数据

In [2]:
# # 加载价格数据
# price_df = load_price_data('./market_close.csv')
# print(f"数据形状: {price_df.shape}")
# print(f"日期范围: {price_df.index.min()} 到 {price_df.index.max()}")
# print(f"\n资产列表:")
# for col in price_df.columns:
#     print(f"  - {col}")

# price_df.head()

In [3]:
# 加载价格数据
price_df = pd.read_excel('./market_close.xlsx')
price_df.columns = price_df.iloc[2]
price_df = price_df.iloc[4:]
price_df['日期'] = pd.to_datetime(price_df['日期'])
price_df = price_df.set_index('日期')
print(f"数据形状: {price_df.shape}")
print(f"日期范围: {price_df.index.min()} 到 {price_df.index.max()}")
print(f"\n资产列表:")
for col in price_df.columns:
    print(f"  - {col}")

price_df.head()

数据形状: (2550, 10)
日期范围: 2015-09-01 00:00:00 到 2026-03-06 00:00:00

资产列表:
  - 上证指数
  - 创业板指
  - 纳斯达克指数
  - 道琼斯工业平均
  - 中证转债
  - 中债-商业银行二级资本债券财富(总值)指数
  - 中债-新综合财富(1年以下)指数
  - 中债-新综合财富(1-3年)指数
  - SGE黄金9999
  - ICE布油


2,上证指数,创业板指,纳斯达克指数,道琼斯工业平均,中证转债,中债-商业银行二级资本债券财富(总值)指数,中债-新综合财富(1年以下)指数,中债-新综合财富(1-3年)指数,SGE黄金9999,ICE布油
日期,,,,,,,,,,
2015-09-01,3166.6239,1889.491,29556.128472,102375.19292,291.3647,108.0298,150.7559,161.255,234.6,320.098792
2015-09-02,3160.167,1855.032,30218.897762,104025.844422,289.3253,108.1052,150.7703,161.2732,234.9,331.900323
2015-09-07,3080.4201,1893.521,29782.236928,102385.372992,292.4632,108.0971,150.8428,161.3674,231,314.232128
2015-09-08,3170.4522,2001.156,30622.641327,104957.766252,303.3656,108.0641,150.8641,161.3759,230.58,324.813456
2015-09-09,3243.0889,2071.717,30266.751696,103424.716624,310.0523,107.8927,150.8783,161.3896,231.1,313.260336


## 2. 风险平价策略回测

In [33]:
# 创建风险平价策略 - 测试单种方法
rp_strategy = RiskParityStrategy(
    lookback=120,           # 12日回看窗口
    rebalance_freq='ME',    # ME月末调仓, QE 季末调仓
    method='CDD',         # 使用SLSQP优化方法计算权重,也可以选择 'CDD' 方法
    compare_methods=False   # 关闭方法对比，使用rp_strategy_compare.print_weights_comparison()可以查看权重对比结果
)

# 创建回测引擎
engine = BacktestEngine(
    init_cash=1_000_000,
    freq='1D'
)

# 运行回测
rp_result = engine.run(rp_strategy, price_df)

d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [34]:
# 查看回测统计
print(rp_result.metrics)

rp_result.stats()

{'total_return': np.float64(0.42744997426651077), 'annualized_return': np.float64(0.052260874268823754), 'annualized_volatility': np.float64(0.008777038096727618), 'sharpe_ratio': np.float64(5.80869257819305), 'sortino_ratio': np.float64(10.178747967916118), 'calmar_ratio': np.float64(4.007959665473801), 'max_drawdown': np.float64(-0.013039271507400696), 'omega_ratio': np.float64(2.411993298888117), 'best_trade': None, 'worst_trade': None, 'win_rate': None}


Start                                  2015-09-01 00:00:00
End                                    2026-03-06 00:00:00
Period                                  2550 days 00:00:00
Start Value                                      1000000.0
End Value                                   1427449.974267
Total Return [%]                                 42.744997
Benchmark Return [%]                            144.795482
Max Gross Exposure [%]                               100.0
Total Fees Paid                                        0.0
Max Drawdown [%]                                  1.303927
Max Drawdown Duration                     64 days 00:00:00
Total Trades                                         11078
Total Closed Trades                                  11068
Total Open Trades                                       10
Open Trade PnL                                97180.322968
Win Rate [%]                                     83.194796
Best Trade [%]                                   57.2677

In [35]:
rp_result.weights

2,上证指数,创业板指,纳斯达克指数,道琼斯工业平均,中证转债,中债-商业银行二级资本债券财富(总值)指数,中债-新综合财富(1年以下)指数,中债-新综合财富(1-3年)指数,SGE黄金9999,ICE布油
2016-03-31,0.005002,0.003131,0.006877,0.008234,0.007870,0.083353,0.631237,0.235722,0.013906,0.004667
2016-04-29,0.005285,0.003382,0.007913,0.010052,0.007611,0.076093,0.654327,0.215084,0.015077,0.005175
2016-05-31,0.004904,0.003306,0.007732,0.009984,0.007191,0.079139,0.653684,0.212814,0.015938,0.005307
2016-06-30,0.004601,0.003022,0.007803,0.010538,0.008531,0.083218,0.647290,0.213449,0.017078,0.004469
2016-07-29,0.005642,0.003582,0.009271,0.013080,0.011217,0.080124,0.629850,0.229983,0.012941,0.004309
...,...,...,...,...,...,...,...,...,...,...
2025-11-28,0.006282,0.002824,0.004396,0.008946,0.008564,0.099171,0.596280,0.263709,0.005024,0.004805
2025-12-31,0.005816,0.002602,0.004354,0.008578,0.007921,0.097658,0.607651,0.255887,0.004355,0.005179
2026-01-30,0.005450,0.002656,0.004421,0.007501,0.006091,0.099390,0.588983,0.277445,0.003229,0.004833
2026-02-27,0.005429,0.002883,0.004548,0.008660,0.005414,0.106821,0.560425,0.299640,0.002409,0.003771


In [36]:
# 可视化结果
rp_viz = BacktestVisualizer(rp_result)
rp_viz.print_metrics()
rp_viz.plot_summary()


Risk Parity 策略表现
总收益率: 42.74%
年化收益率: 5.23%
年化波动率: 0.88%
夏普比率: 5.809
索提诺比率: 10.179
Calmar比率: 4.008
Omega比率: 2.412
最大回撤: -1.30%



In [37]:
# 权重热力图
rp_viz.plot_weights_heatmap(freq='QE')

In [38]:
# 权重变化分析与调仓点标注
# 分析模型的重要调仓时机
rp_viz.plot_weight_changes_analysis(threshold=0.03)

In [39]:
# 资产价格走势与权重变化组合分析
# 这个图可以帮助你看到模型何时对各个资产进行加减仓
rp_viz.plot_assets_and_weights(
    price_df,           # 原始价格数据
    freq='W',           # 按周显示权重变化
    top_n=8             # 显示权重最大的8个资产
)

### 如何解读这些图表

**资产走势与权重分析图**：
- **上半部分**：各资产价格走势（归一化到100），帮助理解资产的历史表现
- **下半部分**：对应的权重配置变化，显示模型何时加减仓
- **分析要点**：
  - 当某资产价格下跌时，模型是否增加权重（抄底）或减少权重（止损）
  - 当某资产价格上涨时，模型是否减少权重（获利了结）或增加权重（追涨）
  - 权重变化频率反映策略的调仓灵敏度

**权重变化分析图**：
- 显示所有资产的权重配置时间序列
- 标注重要的调仓点（权重变化超过阈值）
- 统计月度调仓频率，帮助理解策略的活跃度

## 3. 均值方差策略回测

In [40]:
# 创建均值方差策略（最大化夏普比率）
mv_strategy = MeanVarianceStrategy(
    lookback=60,
    rebalance_freq='ME'
)

# 运行回测
mv_result = engine.run(mv_strategy, price_df)

# 可视化
mv_viz = BacktestVisualizer(mv_result)
mv_viz.print_metrics()
mv_viz.plot_summary()

d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`




Mean Variance 策略表现
总收益率: 111.10%
年化收益率: 11.29%
年化波动率: 10.39%
夏普比率: 1.082
索提诺比率: 1.510
Calmar比率: 0.696
Omega比率: 1.182
最大回撤: -16.22%



## 4. 策略对比

In [41]:
# 创建多个策略进行对比
strategies = [
    RiskParityStrategy(lookback=60, rebalance_freq='ME'),
    RiskParityStrategy(lookback=120, rebalance_freq='QE'),
    MeanVarianceStrategy(lookback=60, rebalance_freq='ME'),
]

names = ['风险平价(60日/月)', '风险平价(120日/季)', '均值方差(60日/月)']

# 运行所有策略
results = []
for strategy, name in zip(strategies, names):
    result = engine.run(strategy, price_df)
    results.append(result)
    print(f"{name}: 总收益={result.metrics['total_return']*100:.2f}%, 夏普={result.metrics['sharpe_ratio']:.3f}")

d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



警告: 2019-07-31 00:00:00 优化失败 - 风险平价SLSQP优化失败: Positive directional derivative for linesearch
风险平价(60日/月): 总收益=64.82%, 夏普=1.018


d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



风险平价(120日/季): 总收益=122.45%, 夏普=2.084


d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



均值方差(60日/月): 总收益=111.10%, 夏普=1.082


In [42]:
# 累计收益对比图
BacktestVisualizer.compare_results(results, names=names)

In [43]:
# 指标对比表
comparison_table = BacktestVisualizer.compare_metrics_table(results, names)
comparison_table

,总收益率 (%),年化收益率 (%),年化波动率 (%),夏普比率,索提诺比率,Calmar比率,最大回撤 (%)
风险平价(60日/月),64.818165,7.414154,7.288479,1.017955,1.433227,0.506002,-14.652431
风险平价(120日/季),122.447287,12.124659,5.565107,2.084479,3.457775,1.668619,-7.266284
均值方差(60日/月),111.103627,11.287766,10.389123,1.081679,1.510492,0.695775,-16.223289


## 5. 创建自定义策略

继承 `BaseStrategy` 即可创建自己的策略

In [ ]:
from portfolio_backtest.strategies.base import BaseStrategy

class EqualWeightStrategy(BaseStrategy):
    """等权重策略 - 自定义策略示例"""
    
    def __init__(self, rebalance_freq='ME'):
        super().__init__(name="Equal Weight", rebalance_freq=rebalance_freq)
        self.rebalance_freq = rebalance_freq
    
    def generate_weights(self, price_df, rebalance_mask=None):
        price_df = self.validate_data(price_df)
        
        if rebalance_mask is None:
            rebalance_dates = self.get_rebalance_dates(price_df, self.rebalance_freq)
            rebalance_mask = pd.Series(
                price_df.index.isin(rebalance_dates),
                index=price_df.index
            )
        
        n_assets = price_df.shape[1]
        equal_weight = 1.0 / n_assets
        
        rebalance_dates = price_df.index[rebalance_mask]
        weights_list = [np.full(n_assets, equal_weight) for _ in rebalance_dates]
        
        return pd.DataFrame(
            weights_list,
            index=rebalance_dates,
            columns=price_df.columns
        )

# 使用自定义策略
ew_strategy = EqualWeightStrategy(rebalance_freq='ME')
ew_result = engine.run(ew_strategy, price_df)

ew_viz = BacktestVisualizer(ew_result)
ew_viz.print_metrics()


Equal Weight 策略表现
总收益率: 145.00%
年化收益率: 13.69%
年化波动率: 10.58%
夏普比率: 1.265
索提诺比率: 1.794
Calmar比率: 0.844
Omega比率: 1.209
最大回撤: -16.22%



d:\0信银\26---new\portfolio_backtest\engine\backtest.py:83: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

